# Final Multiscale NLP Correction Summary

This notebook summarizes the final results of the event-augmented port disruption prediction project.

It does not download raw data. It reads saved processed files and reproduces the main result tables and figures.

## Research Question

Can NLP-derived external event signals improve the prediction of abnormal next-week port activity when they are used as a spatially layered correction mechanism on top of an operational baseline?

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

comparison_path = PROCESSED_DIR / "multiscale_two_stage_default_comparison_2021_2025.csv"
threshold_path = PROCESSED_DIR / "multiscale_best_thresholds_2021_2025.csv"

comparison = pd.read_csv(comparison_path)
best_thresholds = pd.read_csv(threshold_path)

display(comparison)
display(best_thresholds)

## Default-Threshold Model Comparison

The main comparison uses 2021-2024 as the training period and 2025 as the test period.

In [ ]:
models_to_plot = [
    "operational_baseline",
    "two_stage_global",
    "two_stage_europe",
    "two_stage_local",
    "two_stage_multiscale_all"
]

model_labels = {
    "operational_baseline": "Baseline",
    "two_stage_global": "Global NLP",
    "two_stage_europe": "Europe NLP",
    "two_stage_local": "Local NLP",
    "two_stage_multiscale_all": "Multiscale NLP"
}

metrics = ["roc_auc", "pr_auc", "f1", "recall", "precision"]

plot_df = comparison[comparison["model"].isin(models_to_plot)].copy()
plot_df["model_label"] = plot_df["model"].map(model_labels)
plot_df = plot_df.set_index("model_label")[metrics]

figure_path = FIGURE_DIR / "model_metric_comparison_multiscale_2021_2025.png"

ax = plot_df.plot(kind="bar", figsize=(13, 7), width=0.8)
plt.title("Model Performance Comparison on 2025 Test Set", fontsize=14)
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(figure_path, dpi=300)
plt.show()

display(plot_df)
print(figure_path)

## Baseline vs Multiscale NLP Correction

This figure focuses on the main baseline-vs-final-model comparison.

In [ ]:
selected_models = ["operational_baseline", "two_stage_multiscale_all"]

model_labels = {
    "operational_baseline": "Operational Baseline",
    "two_stage_multiscale_all": "Multiscale NLP Correction"
}

metric_labels = {
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
    "f1": "F1",
    "recall": "Recall",
    "precision": "Precision"
}

core_df = comparison[comparison["model"].isin(selected_models)].copy()
core_df["model_label"] = core_df["model"].map(model_labels)
core_df = core_df.set_index("model_label")[metrics]
core_df = core_df.rename(columns=metric_labels)

figure_path = FIGURE_DIR / "baseline_vs_multiscale_core_metrics.png"

ax = core_df.T.plot(kind="bar", figsize=(10, 6), width=0.7)
plt.title("Baseline vs Multiscale NLP Correction on 2025 Test Set", fontsize=14)
plt.xlabel("Metric")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Model")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(figure_path, dpi=300)
plt.show()

display(core_df)
print(figure_path)

## Final Interpretation

The multiscale NLP correction model improves the operational baseline across all default-threshold metrics. The result suggests that external news signals are most useful when they are spatially layered and used as a correction mechanism, rather than added as raw global features.